# Vulnerability Management RAG — Colab Simulation

Test-drive the multi-agent RAG vulnerability bot from the [Portfolio repo](https://github.com/Konaguy/Portfolio/tree/claude/ai-secops-rag) end to end, in the browser.

**What this notebook does**
1. Clones the repo and installs the core dependencies.
2. Ingests the bundled sample Tenable export + security policies into an in-memory Qdrant vector store.
3. Runs the LangGraph agent pipeline (router → retriever → specialist) and lets you ask it questions.
4. Simulates an **Azure Defender / MDVM scan** of 2 servers + 2 Windows 11 PCs and asks the bot about the findings.

It runs **fully offline by default** using the deterministic stub LLM and a dependency-free embedder — no API key, no services. Add an Anthropic API key in the optional cell for real Claude-authored answers.

> Runtime → *Run all*, or run the cells top to bottom.


## 1 · Clone the repo and install dependencies

In [ ]:
!git clone --depth 1 --branch claude/ai-secops-rag https://github.com/Konaguy/Portfolio.git 2>/dev/null || echo "already cloned"
%cd Portfolio/vuln-management-rag

# Core deps only. (Chainlit + sentence-transformers are NOT needed for the
# headless simulation; the optional UI cell at the end installs Chainlit.)
!pip install -q "anthropic>=0.40" "langgraph>=0.2.28" "qdrant-client>=1.9" "pydantic>=2.5"
print("\nInstalled. Working dir:")
!pwd


## 2 · (Optional) Anthropic API key

Leave the prompt **blank** to run offline with the deterministic stub LLM (great for seeing the pipeline wiring). Paste a key to get real Claude-authored, grounded answers.

In [ ]:
import os, getpass
key = getpass.getpass("Anthropic API key (optional — press Enter to skip): ").strip()
if key:
    os.environ["ANTHROPIC_API_KEY"] = key
print("LLM:", "Claude (" + os.environ.get("ANTHROPIC_MODEL", "claude-opus-5") + ")" if os.environ.get("ANTHROPIC_API_KEY") else "offline stub (no key)")


## 3 · Build the corpus and the agent graph

Ingests the sample Tenable findings and the two security policies into an in-memory Qdrant collection, then wires up the LangGraph graph.

In [ ]:
import os
from vulnrag.config import Settings
from vulnrag.embeddings import build_embedder
from vulnrag.vectorstore import VectorStore
from vulnrag.ingest import parse_tenable_file, parse_policy_dir
from vulnrag.agents.graph import VulnRagGraph
from vulnrag.llm import ClaudeClient, FakeLLMClient

settings = Settings.from_env()   # defaults: in-memory Qdrant, dependency-free hashing embedder
store = VectorStore(settings, build_embedder(settings))
n_v = store.upsert_vulnerabilities(parse_tenable_file("data/sample_tenable_export.csv"))
n_p = store.upsert_policies(parse_policy_dir("data/sample_policies"))

llm = ClaudeClient(model=settings.anthropic_model) if os.environ.get("ANTHROPIC_API_KEY") else FakeLLMClient()
graph = VulnRagGraph.from_settings(settings, llm, store=store)

print(f"Ingested {n_v} vulnerability findings + {n_p} policy chunks  ->  {store.count()} corpus chunks")


## 4 · Ask the bot

The helper prints the structured answer: summary, ordered remediation steps, the prioritization rationale, the **citations** (chunk IDs it grounded on — a hallucinated citation is rejected upstream), and the agent **trace** (which intent it routed to and how many chunks it retrieved).

In [ ]:
def ask(q):
    final = graph.answer(q)
    a = final["answer"]
    print("Q:", q)
    print("\nSummary:\n ", a["summary"])
    if a["remediation_steps"]:
        print("\nRemediation steps:")
        for i, s in enumerate(a["remediation_steps"], 1):
            print(f"  {i}. {s}")
    if a["prioritization_rationale"]:
        print("\nPrioritization:", a["prioritization_rationale"])
    print("\nCitations:", ", ".join(a["cited_chunk_ids"]) or "(none)")
    print("Trace:   ", " -> ".join(final.get("trace", [])))
    print("=" * 78)

ask("How do I remediate the Log4Shell findings on the DMZ hosts?")
ask("Which of the critical vulnerabilities should I patch first?")
ask("What is our SLA for patching critical vulnerabilities?")


## 5 · Interactive — ask your own questions

Run this cell and type questions (blank line to stop).

In [ ]:
while True:
    q = input("\nAsk the vuln bot (blank to stop): ").strip()
    if not q:
        break
    ask(q)


## 6 · Simulate an Azure Defender / MDVM scan

This is the Azure path: instead of a Tenable CSV, the `FakeDefenderConnector` returns the bundled sample findings for **2 servers + 2 Windows 11 PCs** (the same shape the live Defender for Endpoint API returns). They're mapped into the bot's `Vulnerability` schema, ingested, and become answerable — exactly as a real scan would.

(To point at a *real* Azure tenant instead, set `AZURE_TENANT_ID` / `AZURE_CLIENT_ID` / `AZURE_CLIENT_SECRET`, `pip install requests`, and swap `FakeDefenderConnector` for `DefenderConnector` — see the project README.)

In [ ]:
from vulnrag.connectors import FakeDefenderConnector
from vulnrag.connectors.defender import findings_to_vulnerabilities
from vulnrag.export import build_findings_document

devices = ["srv-web-01", "srv-sql-01", "pc-hr-07", "pc-eng-12"]
conn = FakeDefenderConnector.from_json_file("data/sample_defender_findings.json")
vulns = findings_to_vulnerabilities(conn.get_findings(devices))
store.upsert_vulnerabilities(vulns)   # now queryable by the same bot

doc = build_findings_document(vulns)
print("Simulated Defender scan of", len({v.host for v in vulns}), "devices")
print("  findings :", doc["summary"]["total_findings"])
print("  severity :", doc["summary"]["severity_counts"])
print()
ask("What are the critical findings on srv-web-01 and how do I remediate them?")


## 7 · (Optional, advanced) Launch the real Chainlit chat UI

The cells above are the reliable simulation. If you want the actual chat UI, this starts the Chainlit app and exposes it with a Cloudflare quick tunnel (no account needed). Tunnels in Colab are best-effort — if it doesn't come up, stick with the headless cells above.

In [ ]:
# Optional: run the Chainlit UI and expose it via a cloudflared quick tunnel.
!pip install -q chainlit
import subprocess, time, re, threading

# cloudflared (no login needed for a quick tunnel)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared

subprocess.Popen(["chainlit", "run", "app.py", "--host", "0.0.0.0", "--port", "8000", "--headless"])
time.sleep(10)

proc = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://localhost:8000"],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
def _print_url():
    for line in proc.stdout:
        if "trycloudflare.com" in line:
            m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", line)
            if m:
                print("\n>>> Open the chat UI:", m.group(0), "\n")
threading.Thread(target=_print_url, daemon=True).start()
time.sleep(12)
print("If no URL appeared above, re-run this cell or use the headless cells.")
